# 01 · Data exploration

Pull and inspect the data + assumption layer behind the AI data-center macro stress model.

Everything runs offline from the YAML assumption layer; the API clients (FRED, EIA, SEC EDGAR, FiscalData) enrich the model when keys/network are available and degrade gracefully otherwise.

> Run with the project venv kernel: `uv run jupyter lab` (or point your Jupyter at `.venv`).

In [ ]:
import sys, os
from pathlib import Path
# make the `src` package importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option('display.max_colwidth', 80)
from src import config

## 1. The assumption layer (with provenance)

Every manually estimated value carries `value / unit / confidence / source_note / last_updated / rationale`. Many are **low-confidence scenario assumptions, not facts**.

In [ ]:
from src.data_sources.manual_inputs import list_assumptions, low_confidence_keys, provenance_note
df = list_assumptions()
df

In [ ]:
print('LOW-CONFIDENCE keys (treat as scenario inputs, not facts):')
for k in low_confidence_keys():
    print(' -', k)

In [ ]:
print(provenance_note('private_credit_ai_datacenter_outstanding'))

## 2. Scenarios

In [ ]:
from src.model.scenarios import load_all_scenarios
scenarios = load_all_scenarios()
rows = []
for name, sc in scenarios.items():
    rows.append({
        'scenario': name,
        'sofr_bps': sc.rates.sofr_bps, '10y_bps': sc.rates.dgs10_bps, '30y_bps': sc.rates.dgs30_bps,
        'spread_bps': sc.credit.credit_spread_bps, 'power_pct': sc.power.power_price_pct,
        'util_delta': sc.ai_demand.utilization_delta, 'pc_nav_haircut': sc.private_credit.nav_haircut,
        'securitization_open': sc.credit.securitization_open,
    })
pd.DataFrame(rows).set_index('scenario')

## 3. Entity universe (hyperscalers + DC REITs)

CIKs map to SEC EDGAR; `ai_capex_share_override` is a manual override because AI-specific capex is rarely disclosed.

In [ ]:
eu = config.load_entity_universe()
rows = []
for group in ('hyperscalers', 'datacenter_operators'):
    for c in eu.get(group, []):
        rows.append({'name': c['name'], 'ticker': c['ticker'], 'cik': c['cik'], 'type': c['type'],
                     'ai_capex_share': c.get('ai_capex_share_override', {}).get('value')})
pd.DataFrame(rows)

## 4. Live data (optional — degrades gracefully offline)

If you have a FRED key (or network for the public CSV fallback), this pulls current Treasury/SOFR levels. Otherwise it returns `{}` and the model uses YAML defaults.

In [ ]:
from src.data_sources.fred import FredClient
try:
    levels = FredClient().get_latest_levels()
except Exception as e:
    levels = {}
    print('FRED unavailable:', e)
print('Latest levels (decimals):', levels or '(none — offline / no key; model uses YAML defaults)')

In [ ]:
# Power assumptions by region (used when EIA is unavailable)
pd.Series(config.assumption_value('power_price_base_by_region', {}), name='USD_per_MWh').to_frame()